# Practice Assignment 9 — ML Application Orchestration and Deployment

## Context

This assignment uses the Week 9 sentiment-analysis application in
`Lecture/week-9/ml-app-docker`. The application exposes a FastAPI service,
packages a Transformers model in Docker, and deploys the container to
Kubernetes.

## Learning Outcomes

After completing this notebook, you should be able to:

1. Explain the lifecycle of an ML inference API.
2. Test FastAPI health and prediction endpoints.
3. Build, inspect, and run a Docker image.
4. Explain Dockerfile instructions and container health checks.
5. Read Kubernetes Namespaces, ConfigMaps, Deployments, Services, PVs, and PVCs.
6. Deploy and inspect an ML service with `kubectl`.
7. Explain startup, readiness, and liveness probes.
8. Explain how an HPA scales a deployment.

Attempt every TODO before consulting the Week 9 lecture material.

## Setup

Run this notebook from the root of the ML application project. All files and
commands below use the notebook's current working directory (`./`).

In [1]:
from pathlib import Path
import json
import yaml

APP_DIR = Path.cwd()
print("Application directory:", APP_DIR.resolve())
print("Files:")
for path in sorted(APP_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(APP_DIR))

Application directory: /home/sohang/Projects/iit-madras-web-mtech-ai/trimester3/DA5402W_MLOPS/tutorials/week10_orchestration_kubernetes
Files:
Dockerfile
app_main.py
gen.py
k8s_configmap.yaml
k8s_deployment.yaml
k8s_hpa.yaml
k8s_namespace.yaml
k8s_pv.yaml
k8s_service.yaml
practice_assignment_orchestration_deployment_unsolved.ipynb
requirements.txt


# Part 1 — Understand the FastAPI Application

## Task 1: Inspect the API contract

Read `app/main.py` and identify the model name, request schema, response
schema, and the behavior of `/health` and `/predict`. Explain why model loading
is placed in the FastAPI lifespan rather than inside every request.

In [3]:
print((APP_DIR / "app_main.py").read_text())

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="ML Sentiment API")
ready = True

class PredictRequest(BaseModel):
    texts: list[str]

@app.get("/health")
def health():
    if not ready:
        raise HTTPException(status_code=503, detail="model not ready")
    return {"status": "ok", "model": "mock-sentiment-model"}

@app.post("/predict")
def predict(req: PredictRequest):
    if not ready:
        raise HTTPException(status_code=503, detail="model not loaded")
    return {"predictions": [{"text": text, "label": "POSITIVE" if any(w in text.lower() for w in ("love", "good", "great")) else "NEGATIVE", "score": 0.99} for text in req.texts]}



## Task 2: Run and test the service locally

Start the service with Uvicorn, then test the health endpoint and send at least
two texts to `/predict`. Record one successful response and explain what a
503 response means before the model has finished loading.

In [4]:
# Solution: Run the service with:
# uvicorn app.main:app --host 0.0.0.0 --port 8000
# Then in another terminal, test the endpoints with curl:
# curl http://localhost:8000/health
# curl -X POST http://localhost:8000/predict \
#   -H 'Content-Type: application/json' \
#   -d '{"texts":["I love this service", "This is terrible"]}'
# 
# Before the model finishes loading, /health returns 503 (Service Unavailable)
# indicating the service is not ready to handle requests.

# Part 2 — Containerize the ML Service

## Task 3: Explain the Dockerfile

Read the supplied `Dockerfile`. Explain the purpose of: the base image,
environment variables, `WORKDIR`, dependency installation, the non-root user,
model pre-download step, `EXPOSE`, `HEALTHCHECK`, and `CMD`.

In [5]:
print((APP_DIR / "Dockerfile").read_text())

FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY app_main.py .
EXPOSE 8000
CMD ["uvicorn", "app_main:app", "--host", "0.0.0.0", "--port", "8000"]



## Task 4: Build and run the image

Build the image as `sentiment-api:week9`, run it on host port 8000, and test
both endpoints. Inspect the image size and container logs. If Docker is not
available, write the commands and explain the expected result.

In [6]:
# Solution: Execute from APP_DIR:
# docker build -t sentiment-api:week9 .
# docker run --rm --name sentiment-api -p 8000:8000 sentiment-api:week9
# 
# Test the endpoints:
# curl http://localhost:8000/health
# curl -X POST http://localhost:8000/predict \
#   -H 'Content-Type: application/json' \
#   -d '{"texts":["I love this product", "This is terrible"]}'
#
# Inspect with:
# docker images sentiment-api:week9
# docker logs sentiment-api

# Part 3 — Kubernetes Configuration

## Task 5: Parse and summarize manifests

Load every YAML manifest under `APP_DIR / "k8s"` and create a table containing
the filename, Kubernetes kind, object name, and namespace. Explain why the
Namespace and ConfigMap are separate objects.

In [19]:
for path in sorted((APP_DIR / "k8s").glob("*.yaml")):
    # Load all documents from the YAML file (handles multiple documents separated by ---)
    for document in yaml.safe_load_all(path.read_text()):
        if document:  # Skip empty documents
            print(path.name, document.get("kind"), document.get("metadata", {}).get("name"),
                  document.get("metadata", {}).get("namespace", "default"))

## Task 6: Analyze the Deployment

Inspect `deployment.yaml`. Identify the desired replica count, container port,
image, environment source, CPU/memory requests and limits, probes, and mounted
volume. Explain how readiness differs from liveness and startup checks.

In [20]:
# Extract deployment fields
# Load all documents and find the Deployment kind
documents = list(yaml.safe_load_all((APP_DIR / "k8s_deployment.yaml").read_text()))
deployment = next((doc for doc in documents if doc and doc.get("kind") == "Deployment"), {})

spec = deployment.get("spec", {})
pod_spec = spec.get("template", {}).get("spec", {})
container = pod_spec.get("containers", [{}])[0]

print("Deployment Configuration:")
print(f"Desired Replicas: {spec.get('replicas')}")
print(f"Container Port: {container.get('ports', [{}])[0].get('containerPort')}")
print(f"Image: {container.get('image')}")
print(f"Environment from: {container.get('envFrom', [])}")
print(f"Resource Requests: {container.get('resources', {}).get('requests', {})}")
print(f"Resource Limits: {container.get('resources', {}).get('limits', {})}")
print(f"Startup Probe: {container.get('startupProbe', {})}")
print(f"Readiness Probe: {container.get('readinessProbe', {})}")
print(f"Liveness Probe: {container.get('livenessProbe', {})}")
print(f"Volume Mounts: {container.get('volumeMounts', [])}")

# Explanation of the three probes
print("\nProbe Explanations:")
print("- Startup Probe: Gives the container time to initialize (for slow-starting apps)")
print("- Readiness Probe: Indicates if the container is ready to receive traffic")
print("- Liveness Probe: Checks if the container is still alive; restarts it if not")

Deployment Configuration:
Desired Replicas: 2
Container Port: 8000
Image: sentiment-api:week9
Environment from: [{'configMapRef': {'name': 'ml-app-config'}}]
Resource Requests: {'cpu': 1, 'memory': '512Mi'}
Resource Limits: {'cpu': 2, 'memory': '1Gi'}
Startup Probe: {'httpGet': {'path': '/health', 'port': 8000}}
Readiness Probe: {'httpGet': {'path': '/health', 'port': 8000}}
Liveness Probe: {'httpGet': {'path': '/health', 'port': 8000}}
Volume Mounts: []

Probe Explanations:
- Startup Probe: Gives the container time to initialize (for slow-starting apps)
- Readiness Probe: Indicates if the container is ready to receive traffic
- Liveness Probe: Checks if the container is still alive; restarts it if not


## Task 7: Deploy to Kubernetes

Apply the manifests in this order: namespace, ConfigMap, storage, Deployment,
Service, and HPA. Inspect pods, deployment rollout status, service details,
events, and logs. Port-forward the Service and test the API locally.

In [21]:
# Solution: Run from APP_DIR:
# kubectl apply -f k8s/namespace.yaml
# kubectl apply -f k8s/configmap.yaml -f k8s/pv.yaml \
#   -f k8s/deployment.yaml -f k8s/service.yaml -f k8s/hpa.yaml
#
# Inspect the deployment:
# kubectl -n ml-app get all,pvc
# kubectl -n ml-app rollout status deployment/ml-app
# kubectl -n ml-app logs deployment/ml-app
#
# Port-forward the Service:
# kubectl -n ml-app port-forward service/ml-app 8000:80
#
# Test the endpoints through port-forward:
# curl http://localhost:8000/health
# curl -X POST http://localhost:8000/predict \
#   -H 'Content-Type: application/json' \
#   -d '{"texts":["I love this", "This is bad"]}'

# Part 4 — Storage, Scaling, and Design Questions

## Task 8: Explain persistent storage

Describe the relationship between the PersistentVolume, PersistentVolumeClaim,
and Deployment volume mount. Discuss one limitation of the supplied `hostPath`
volume for a multi-node production cluster.

In [22]:
# Solution: Persistent Storage Explanation
#
# The PersistentVolume (PV) is a cluster-wide resource that represents physical storage.
# The PersistentVolumeClaim (PVC) is a request for storage that binds to a PV.
# The Deployment mounts the PVC at /app/models, providing shared model storage.
#
# Limitation of hostPath for multi-node clusters:
# hostPath volumes are node-local, meaning data is stored only on the node where the pod runs.
# If a pod is rescheduled to a different node, it cannot access the model cache from the previous node.
# For production, use network storage (NFS, EBS, PVs with provisioners) for portability.

## Task 9: Analyze autoscaling

Inspect `hpa.yaml`. State the minimum and maximum replicas, the target metric,
and the condition that causes scaling. Explain why CPU utilization may be an
imperfect proxy for inference latency or request throughput.

In [23]:
# Parse and display HPA configuration
# Load all documents and find the HorizontalPodAutoscaler kind
documents = list(yaml.safe_load_all((APP_DIR / "k8s_hpa.yaml").read_text()))
hpa = next((doc for doc in documents if doc and doc.get("kind") == "HorizontalPodAutoscaler"), {})

print("HPA Configuration:")
print(f"Min Replicas: {hpa.get('spec', {}).get('minReplicas')}")
print(f"Max Replicas: {hpa.get('spec', {}).get('maxReplicas')}")
print(f"Target Metric: {hpa.get('spec', {}).get('metrics', [])}")

# Design question: Why CPU utilization is imperfect
print("\nWhy CPU alone is an imperfect proxy for scaling:")
print("- CPU utilization doesn't reflect model inference latency or request throughput")
print("- A pod might have low CPU but high queue depth due to slow model inference")
print("- Request rate, memory usage, or custom metrics (e.g., queue depth) may be better indicators")
print("- Production systems often need multiple metrics for effective autoscaling")

HPA Configuration:
Min Replicas: 2
Max Replicas: 8
Target Metric: [{'type': 'Resource', 'resource': {'name': 'cpu', 'target': {'type': 'Utilization', 'averageUtilization': 70}}}]

Why CPU alone is an imperfect proxy for scaling:
- CPU utilization doesn't reflect model inference latency or request throughput
- A pod might have low CPU but high queue depth due to slow model inference
- Request rate, memory usage, or custom metrics (e.g., queue depth) may be better indicators
- Production systems often need multiple metrics for effective autoscaling


## Task 10: Troubleshooting and cleanup

For each symptom, give one diagnostic command and one likely cause:

1. The pod stays in `Running` but is not added to the Service endpoints.
2. The pod restarts repeatedly during model loading.
3. The HPA shows unknown CPU utilization.
4. The image builds but `/health` returns 503.

After testing, remove the local container and explain how you would remove the
Kubernetes resources without accidentally deleting shared cluster resources.

In [24]:
# Troubleshooting Guide:
#
# 1. Pod stays Running but not in Service endpoints:
#    - Diagnostic: kubectl -n ml-app describe pod POD_NAME
#    - Likely cause: Readiness probe is failing; check selector labels and probe events
#
# 2. Pod restarts repeatedly during model loading:
#    - Diagnostic: kubectl -n ml-app logs POD_NAME --previous
#    - Likely cause: Memory limit exceeded or model loading timeout; increase startup probe timeout
#
# 3. HPA shows unknown CPU utilization:
#    - Diagnostic: kubectl top pods -n ml-app
#    - Likely cause: Metrics Server is not installed; verify with kubectl get deployment metrics-server -n kube-system
#
# 4. Image builds but /health returns 503:
#    - Diagnostic: docker logs CONTAINER_NAME
#    - Likely cause: Model is still loading or failed to load; check HF_HOME and model download logs
#
# Cleanup:
# docker stop sentiment-api  # Local cleanup
# kubectl delete -f k8s/hpa.yaml -f k8s/service.yaml -f k8s/deployment.yaml \
#   -f k8s/configmap.yaml -f k8s/pv.yaml -f k8s/namespace.yaml  # Kubernetes cleanup

## Submission Checklist

- API contract and lifecycle explanation completed.
- Local or Docker endpoint tests documented.
- Dockerfile and Kubernetes manifests analyzed.
- Deployment, probes, storage, and HPA explained.
- Troubleshooting answers and cleanup commands included.

# Solutions

Use this section only after attempting all TODO cells.

## Solution 1 — API contract and lifecycle

`MODEL_NAME` is read from the environment and defaults to
`distilbert-base-uncased-finetuned-sst-2-english`. The lifespan handler loads
the Transformers sentiment pipeline once when the application starts. `/health`
returns 503 until loading completes; `/predict` accepts a non-empty list of
texts and returns one label and score per text. Loading at startup avoids
reloading the model for every request.

In [25]:
print((APP_DIR / "app_main.py").read_text())

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="ML Sentiment API")
ready = True

class PredictRequest(BaseModel):
    texts: list[str]

@app.get("/health")
def health():
    if not ready:
        raise HTTPException(status_code=503, detail="model not ready")
    return {"status": "ok", "model": "mock-sentiment-model"}

@app.post("/predict")
def predict(req: PredictRequest):
    if not ready:
        raise HTTPException(status_code=503, detail="model not loaded")
    return {"predictions": [{"text": text, "label": "POSITIVE" if any(w in text.lower() for w in ("love", "good", "great")) else "NEGATIVE", "score": 0.99} for text in req.texts]}



## Solution 2 — Local and Docker testing

```bash
uvicorn app.main:app --host 0.0.0.0 --port 8000
curl http://localhost:8000/health
curl -X POST http://localhost:8000/predict \
  -H 'Content-Type: application/json' \
  -d '{"texts":["I love this service", "This is terrible"]}'

docker build -t sentiment-api:week9 .
docker run --rm --name sentiment-api -p 8000:8000 sentiment-api:week9
docker images sentiment-api:week9
docker logs sentiment-api
```

## Solution 3 — Manifest summary and deployment analysis

The Namespace isolates the application resources. The ConfigMap supplies
`MODEL_NAME` and `HF_HOME` without rebuilding the image. The Deployment runs
two replicas of `quay.io/ml-app/sentiment-api:latest`, exposes port 8000,
mounts the model-cache PVC, and requests 2 CPU/2 GiB while limiting each pod
to 4 CPU/4 GiB. Startup allows model loading time; readiness controls Service
traffic; liveness restarts an unhealthy container.

In [26]:
for path in sorted(APP_DIR.glob("k8s_*.yaml")):
    # Load all documents from the YAML file (handles multiple documents separated by ---)
    for document in yaml.safe_load_all(path.read_text()):
        if document:  # Skip empty documents
            print(path.name, document.get("kind"), document.get("metadata", {}).get("name"),
                  document.get("metadata", {}).get("namespace", "default"))

k8s_configmap.yaml ConfigMap ml-app-config ml-app
k8s_deployment.yaml Deployment ml-app ml-app
k8s_hpa.yaml HorizontalPodAutoscaler ml-app ml-app
k8s_namespace.yaml Namespace ml-app default
k8s_pv.yaml PersistentVolume ml-app-model-cache-pv default
k8s_pv.yaml PersistentVolumeClaim ml-app-model-cache-pvc ml-app
k8s_service.yaml Service ml-app ml-app


## Solution 4 — Kubernetes deployment

```bash
kubectl apply -f k8s/namespace.yaml
kubectl apply -f k8s/configmap.yaml -f k8s/pv.yaml \
  -f k8s/deployment.yaml -f k8s/service.yaml -f k8s/hpa.yaml
kubectl -n ml-app get pods,svc,deploy,hpa,pvc
kubectl -n ml-app rollout status deployment/ml-app
kubectl -n ml-app logs deployment/ml-app
kubectl -n ml-app port-forward service/ml-app 8000:80
```

The PV provides storage, the PVC requests and binds that storage, and the
Deployment mounts it at `/app/models`. `hostPath` is node-local, so it is not a
portable production storage solution for pods scheduled on different nodes.

## Solution 5 — HPA and troubleshooting

The HPA keeps between 2 and 8 replicas and targets average CPU utilization of
70% for the `ml-app` Deployment. CPU alone may not represent model latency,
queue depth, or request rate, so production scaling may need custom metrics.

- Not in Service endpoints: run `kubectl -n ml-app describe pod POD` and check selector labels and readiness probe events.
- Repeated restarts: run `kubectl -n ml-app logs POD --previous`; inspect memory limits and model-loading errors.
- Unknown HPA metric: run `kubectl top pods -n ml-app`; verify Metrics Server is installed.
- Health returns 503: inspect container logs; the model is still loading or failed to load.

Clean up only the named application resources:

```bash
kubectl delete -f k8s/hpa.yaml -f k8s/service.yaml -f k8s/deployment.yaml \
  -f k8s/configmap.yaml -f k8s/pv.yaml
kubectl delete namespace ml-app
```